# Integrated Conversational Medical Dataset
## End-to-End Audit · Sanitization · Structural Harmonization · Validation

**Input** : `integrated_conversational_backbone.csv`  
**Output** : `data/processed/medical_conversations_clean.csv`  

| Step | Description |
|------|-------------|
| 1 | Load & Verify |
| 2 | Column Normalisation |
| 3 | Text Sanitisation |
| 4 | Null & Dialogue Integrity Cleaning |
| 5 | Structural Re-indexing |
| 6 | Save Output |
| 7 | Automated Validation Suite |

## 0 · Imports & Path Resolution

In [1]:
import os
import re
import sys
import unicodedata
import logging
from dataclasses import dataclass, field
from typing import List, Tuple

import pandas as pd

# ── Path resolver: Kaggle-first, falls back to local ──────────────────────
KAGGLE_INPUT  = "/kaggle/input"
KAGGLE_OUTPUT = "/kaggle/working"
LOCAL_INPUT   = os.path.join(os.getcwd(), "..", "data", "raw")
LOCAL_OUTPUT  = os.path.join(os.getcwd(), "..", "data", "processed")

def _resolve_paths() -> Tuple[str, str]:
    if os.path.isdir(KAGGLE_INPUT):
        for sub in os.listdir(KAGGLE_INPUT):
            candidate = os.path.join(KAGGLE_INPUT, sub,
                                     "integrated_conversational_backbone.csv")
            if os.path.isfile(candidate):
                return os.path.dirname(candidate), KAGGLE_OUTPUT
        return KAGGLE_INPUT, KAGGLE_OUTPUT
    else:
        os.makedirs(LOCAL_OUTPUT, exist_ok=True)
        return LOCAL_INPUT, LOCAL_OUTPUT

INPUT_DIR, OUTPUT_DIR = _resolve_paths()
INPUT_FILE  = os.path.join(INPUT_DIR,  "integrated_conversational_backbone.csv")
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "medical_conversations_clean.csv")

print(f"Input  : {INPUT_FILE}")
print(f"Output : {OUTPUT_FILE}")

Input  : C:\Users\nirmi\Desktop\Capstone\notebooks\..\data\raw\integrated_conversational_backbone.csv
Output : C:\Users\nirmi\Desktop\Capstone\notebooks\..\data\processed\medical_conversations_clean.csv


## 0.1 · Logging & Shared Constants

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("audit")

DIVIDER = "-" * 72

# Raw columns to retain before rename
RAW_KEEP_COLS = ["dialogue_id", "turn_id", "speaker", "utterance", "source_dataset"]

# Canonical output schema
CANONICAL_COLS = ["conversation_id", "turn_id", "role", "utterance", "source_dataset"]

# Raw source_dataset  →  canonical label
SOURCE_LABEL_MAP = {
    "MedDialog":           "MedDialog-EN",
    "HealthChat-LMSYS":    "HealthChat-11k",
    "HealthChat-WildChat": "HealthChat-11k",
}

# Raw speaker/role  →  canonical role
ROLE_NORMALISE_MAP = {
    "user":      "user",
    "patient":   "user",
    "human":     "user",
    "customer":  "user",
    "assistant": "assistant",
    "doctor":    "assistant",
    "physician": "assistant",
    "bot":       "assistant",
    "agent":     "assistant",
}

# conversation_id prefix per raw source
ID_PREFIX_MAP = {
    "MedDialog":           "meddialog",
    "HealthChat-LMSYS":    "healthchat",
    "HealthChat-WildChat": "healthchat",
}

# Compiled regex patterns for text sanitisation
_HTML_RE  = re.compile(r"<[^>]+>")
_CTRL_RE  = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")
_SPACE_RE = re.compile(r"[ \t\r\n]+")

print("Constants loaded.")

Constants loaded.


## 0.2 · AuditReport & CheckResult Dataclasses

In [3]:
@dataclass
class AuditReport:
    raw_rows:               int = 0
    raw_conversations:      int = 0
    removed_null_terminal:  int = 0
    removed_null_mid_conv:  int = 0
    removed_null_user_turns:int = 0
    removed_assistant_first:int = 0
    removed_hanging_user:   int = 0
    clean_conversations:    int = 0
    clean_rows:             int = 0
    source_breakdown:       dict = field(default_factory=dict)

    def banner(self) -> str:
        lines = [
            DIVIDER, "  AUDIT SUMMARY", DIVIDER,
            f"  Raw rows loaded          : {self.raw_rows:>10,}",
            f"  Raw conversations        : {self.raw_conversations:>10,}",
            "",
            "  REMOVALS",
            f"  - Null terminal asst turn (rows trimmed)  : {self.removed_null_terminal:>7,}",
            f"  - Null mid-turn (convos dropped)          : {self.removed_null_mid_conv:>7,}",
            f"  - Null user turn (convos dropped)         : {self.removed_null_user_turns:>7,}",
            f"  - Asst-first convo (convos dropped)       : {self.removed_assistant_first:>7,}",
            f"  - Hanging user final turn (convos dropped): {self.removed_hanging_user:>7,}",
            "",
            "  OUTPUT",
            f"  Clean conversations      : {self.clean_conversations:>10,}",
            f"  Clean rows (turns)       : {self.clean_rows:>10,}",
            "",
            "  SOURCE BREAKDOWN",
        ]
        for src, cnt in self.source_breakdown.items():
            lines.append(f"    {src:<20} : {cnt:>8,} conversations")
        lines.append(DIVIDER)
        return "\n".join(lines)


@dataclass
class CheckResult:
    name:   str
    passed: bool
    detail: str = ""

    def __str__(self):
        status = "[PASS]" if self.passed else "[FAIL]"
        base   = f"  {status}  {self.name}"
        return base if not self.detail else f"{base}\n         -> {self.detail}"


# Instantiate report; it will be populated across steps
report = AuditReport()
print("Dataclasses ready.")

Dataclasses ready.


---
## Step 1 · Load & Verify

In [4]:
log.info(DIVIDER)
log.info("  STEP 1/7 -- Load & Verify")
log.info(DIVIDER)

df = pd.read_csv(INPUT_FILE, low_memory=False, dtype={"original_id": str})

log.info("Raw shape : %s rows x %s columns", *df.shape)
log.info("Columns   : %s", df.columns.tolist())

# Schema check
missing = [c for c in RAW_KEEP_COLS if c not in df.columns]
if missing:
    raise ValueError(f"[SCHEMA ERROR] Missing expected columns: {missing}")
log.info("[PASS] All required source columns present: %s", RAW_KEEP_COLS)

# Source label check
found_sources = set(df["source_dataset"].dropna().unique())
unknown_sources = found_sources - set(SOURCE_LABEL_MAP.keys())
if unknown_sources:
    log.warning("[WARN] Unrecognised source_dataset values: %s", unknown_sources)
else:
    log.info("[PASS] All source_dataset values recognised: %s", sorted(found_sources))

report.raw_rows          = len(df)
report.raw_conversations = df["dialogue_id"].nunique()
log.info("Raw conversations : %d", report.raw_conversations)

# Strip redundant columns immediately
df = df[RAW_KEEP_COLS].copy()
log.info("[PASS] Stripped to minimal schema: %s", RAW_KEEP_COLS)

print("\nRaw null counts:")
display(df.isnull().sum().to_frame("nulls").T)
print("\nsource_dataset distribution:")
display(df["source_dataset"].value_counts().to_frame())
print("\nspeaker distribution:")
display(df["speaker"].value_counts().to_frame())

19:43:15  INFO      ------------------------------------------------------------------------


19:43:15  INFO        STEP 1/7 -- Load & Verify


19:43:15  INFO      ------------------------------------------------------------------------


19:43:16  INFO      Raw shape : 276172 rows x 8 columns


19:43:16  INFO      Columns   : ['dialogue_id', 'turn_id', 'speaker', 'utterance', 'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin']


...

,dialogue_id,turn_id,speaker,utterance,source_dataset
nulls,0,0,0,198,0



source_dataset distribution:


,count
source_dataset,
MedDialog,224328
HealthChat-LMSYS,28614
HealthChat-WildChat,23230



speaker distribution:


,count
speaker,
user,138086
assistant,138086


---
## Step 2 · Column Normalisation

In [5]:
log.info(DIVIDER)
log.info("  STEP 2/7 -- Column Normalisation")
log.info(DIVIDER)
log.info("Normalising columns ...")

# Build prefixed conversation_id to prevent cross-source collision
prefix_series = df["source_dataset"].map(ID_PREFIX_MAP).fillna("unknown")
df["conversation_id"] = prefix_series + "_" + df["dialogue_id"].astype(str)

# Normalise role
df["role"] = (
    df["speaker"]
    .str.strip()
    .str.lower()
    .map(ROLE_NORMALISE_MAP)
)
unmapped_roles = df["role"].isnull().sum()
if unmapped_roles > 0:
    log.warning("[WARN] %d speaker values unmapped -- will be removed later.",
                unmapped_roles)

# Canonicalise source_dataset label
df["source_dataset"] = df["source_dataset"].map(SOURCE_LABEL_MAP)

# Drop legacy columns
df = df[CANONICAL_COLS].copy()

log.info("[PASS] Column normalisation complete. Schema: %s", CANONICAL_COLS)

print("\nSample after normalisation:")
display(df.head(6))

19:43:16  INFO      ------------------------------------------------------------------------


19:43:16  INFO        STEP 2/7 -- Column Normalisation


19:43:16  INFO      ------------------------------------------------------------------------


19:43:16  INFO      Normalising columns ...


19:43:17  INFO      [PASS] Column normalisation complete. Schema: ['conversation_id', 'turn_id', 'role', 'utterance', 'source_dataset']



Sample after normalisation:


,conversation_id,turn_id,role,utterance,source_dataset
0,healthchat_000573958699464e9de6493b5e182fab,0,user,Prompt: I want you to be my personal mental he...,HealthChat-11k
1,healthchat_000573958699464e9de6493b5e182fab,1,assistant,"NAME_2, it's nice to meet you. I'm here to hel...",HealthChat-11k
2,healthchat_000573958699464e9de6493b5e182fab,2,user,I guess there's quite a few things. But I prim...,HealthChat-11k
3,healthchat_000573958699464e9de6493b5e182fab,3,assistant,"Okay, that's a common theme for many people wi...",HealthChat-11k
4,healthchat_000573958699464e9de6493b5e182fab,4,user,"Yeah, for example today I worked out in the mo...",HealthChat-11k
5,healthchat_000573958699464e9de6493b5e182fab,5,assistant,I see. It sounds like you're experiencing exce...,HealthChat-11k


---
## Step 3 · Text Sanitisation

In [6]:
log.info(DIVIDER)
log.info("  STEP 3/7 -- Text Sanitisation")
log.info(DIVIDER)

def sanitise_text(text) -> str:
    """NFKC normalisation, strip HTML, remove control chars, collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = _HTML_RE.sub(" ", text)
    text = _CTRL_RE.sub("", text)
    text = _SPACE_RE.sub(" ", text)
    return text.strip()

log.info("Sanitising utterance text ...")
df["utterance"] = df["utterance"].apply(sanitise_text)
# Empty strings -> NaN so null-handling logic applies uniformly
df["utterance"] = df["utterance"].replace("", pd.NA)

log.info("[PASS] Text sanitisation applied.")
print(f"Null utterances after sanitisation: {df['utterance'].isna().sum()}")

19:43:17  INFO      ------------------------------------------------------------------------


19:43:17  INFO        STEP 3/7 -- Text Sanitisation


19:43:17  INFO      ------------------------------------------------------------------------


19:43:17  INFO      Sanitising utterance text ...


19:43:21  INFO      [PASS] Text sanitisation applied.


Null utterances after sanitisation: 198


---
## Step 4 · Null & Dialogue Integrity Cleaning

In [7]:
log.info(DIVIDER)
log.info("  STEP 4/7 -- Null & Dialogue Integrity Cleaning")
log.info(DIVIDER)
log.info("Resolving nulls and dialogue integrity ...")

# Sort deterministically
df = df.sort_values(["conversation_id", "turn_id"]).reset_index(drop=True)

# ── A. Terminal null assistant turns → trim the single turn ───────────────
last_idx           = df.groupby("conversation_id")["turn_id"].transform("max")
is_last            = df["turn_id"] == last_idx
terminal_null_asst = is_last & df["utterance"].isna() & (df["role"] == "assistant")

report.removed_null_terminal = int(terminal_null_asst.sum())
log.info("  Trimming %d terminal null assistant turns ...", report.removed_null_terminal)
df = df[~terminal_null_asst].copy()

# ── B. Null user turns → drop entire conversation ─────────────────────────
null_user_convos = df.loc[
    df["utterance"].isna() & (df["role"] == "user"), "conversation_id"
].unique()
report.removed_null_user_turns = len(null_user_convos)
log.info("  Dropping %d conversations with null user turns ...",
         report.removed_null_user_turns)
df = df[~df["conversation_id"].isin(null_user_convos)].copy()

# ── C. Remaining nulls = mid-dialogue assistant → drop conversation ────────
remaining_null_convos = df.loc[df["utterance"].isna(), "conversation_id"].unique()
report.removed_null_mid_conv = len(remaining_null_convos)
log.info("  Dropping %d conversations with mid-dialogue null assistant turns ...",
         report.removed_null_mid_conv)
df = df[~df["conversation_id"].isin(remaining_null_convos)].copy()

# ── D. Conversations starting with assistant → drop ───────────────────────
first_roles       = (df.sort_values(["conversation_id", "turn_id"])
                       .groupby("conversation_id")["role"].first())
asst_first_convos = first_roles[first_roles != "user"].index
report.removed_assistant_first = len(asst_first_convos)
log.info("  Dropping %d conversations starting with assistant ...",
         report.removed_assistant_first)
df = df[~df["conversation_id"].isin(asst_first_convos)].copy()

# ── E. Hanging unanswered final user turns → drop conversation ────────────
last_roles = (df.sort_values(["conversation_id", "turn_id"])
                .groupby("conversation_id")["role"].last())
hanging_user_convos = last_roles[last_roles != "assistant"].index
report.removed_hanging_user = len(hanging_user_convos)
log.info("  Dropping %d conversations with hanging final user turn ...",
         report.removed_hanging_user)
df = df[~df["conversation_id"].isin(hanging_user_convos)].copy()

log.info("[PASS] Null & integrity cleaning complete. Rows remaining: %d", len(df))

print(f"\nRows remaining : {len(df):,}")
print(f"Null utterances: {df['utterance'].isna().sum()}")

19:43:21  INFO      ------------------------------------------------------------------------


19:43:21  INFO        STEP 4/7 -- Null & Dialogue Integrity Cleaning


19:43:21  INFO      ------------------------------------------------------------------------


19:43:21  INFO      Resolving nulls and dialogue integrity ...


19:43:21  INFO        Trimming 93 terminal null assistant turns ...


...

19:43:22  INFO        Dropping 37 conversations with mid-dialogue null assistant turns ...


19:43:22  INFO        Dropping 0 conversations starting with assistant ...


19:43:22  INFO        Dropping 60 conversations with hanging final user turn ...


19:43:22  INFO      [PASS] Null & integrity cleaning complete. Rows remaining: 274162



Rows remaining : 274,162
Null utterances: 0


---
## Step 5 · Structural Re-indexing

In [8]:
log.info(DIVIDER)
log.info("  STEP 5/7 -- Structural Re-indexing")
log.info(DIVIDER)
log.info("Re-indexing turn_ids ...")

df = df.sort_values(["conversation_id", "turn_id"]).copy()

# cumcount gives guaranteed 0, 1, 2, ... per group
df["turn_id"] = df.groupby("conversation_id").cumcount().astype(int)

df = df[CANONICAL_COLS].reset_index(drop=True)

log.info("[PASS] turn_id re-indexed.")

# Preview first conversation
sample_id = df["conversation_id"].iloc[0]
print(f"\nSample conversation: {sample_id}")
display(df[df["conversation_id"] == sample_id][["turn_id", "role", "utterance"]]
          .head(6)
          .style.set_properties(**{"text-align": "left"}))

19:43:22  INFO      ------------------------------------------------------------------------


19:43:22  INFO        STEP 5/7 -- Structural Re-indexing


19:43:22  INFO      ------------------------------------------------------------------------


19:43:22  INFO      Re-indexing turn_ids ...


19:43:22  INFO      [PASS] turn_id re-indexed.



Sample conversation: healthchat_000573958699464e9de6493b5e182fab


,turn_id,role,utterance
0,0,user,"Prompt: I want you to be my personal mental health Al counselor. I am a 16 year old male, struggling with OCD. Let's have a counseling session. I want you to utilize evidence-based therapeutic techniques to guide me through my struggles and help me improve my mental health. Remember we are having a chat -- don't overwhelm me with a lot of text and keep the conversation flowing. Input: ""Hey there, I'm NAME_1"""
1,1,assistant,"NAME_2, it's nice to meet you. I'm here to help you with your mental health, specifically with your OCD. Let's start by talking about what you're currently experiencing with your OCD. What are some of the thoughts or behaviors that are causing you distress?"
2,2,user,I guess there's quite a few things. But I primarily struggle with cleanliness
3,3,assistant,"Okay, that's a common theme for many people with OCD. Can you tell me more about what you do to try and maintain cleanliness? Are there certain things or situations that trigger your OCD thoughts and behaviors?"
4,4,user,"Yeah, for example today I worked out in the morning and wanted to wash my hands after. But I didn't feel like they were clean enough. So I rewashed them over 15 times over the span of an hour, until my skin was completely dry."
5,5,assistant,"I see. It sounds like you're experiencing excessive hand-washing, which is a common symptom of OCD. It can be difficult to control these compulsions, but there are ways to manage them. One technique that may be helpful for you is called ""stimulus control."" It involves breaking the connection between the trigger and the compulsive behavior. For example, you can set a timer for hand-washing and only wash your hands for a certain amount of time, or you can use a specific type of soap that you only use for washing your hands. By breaking the connection between the trigger (dirt or discomfort) and the compulsive behavior (excessive washing), you can gradually weaken the OCD symptoms. Do you think this technique might be helpful for you? Are there any other compulsions or thoughts that are causing you distress?"


---
## Step 6 · Save Output

In [9]:
log.info(DIVIDER)
log.info("  STEP 6/7 -- Save Output")
log.info(DIVIDER)

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

size_mb = os.path.getsize(OUTPUT_FILE) / (1024 ** 2)
log.info("[PASS] Output saved -> %s  (%.2f MB, %d rows)", OUTPUT_FILE, size_mb, len(df))

print(f"\nSaved to : {OUTPUT_FILE}")
print(f"Size     : {size_mb:.2f} MB")
print(f"Rows     : {len(df):,}")
print(f"Columns  : {df.columns.tolist()}")

19:43:22  INFO      ------------------------------------------------------------------------


19:43:22  INFO        STEP 6/7 -- Save Output


19:43:22  INFO      ------------------------------------------------------------------------


19:43:24  INFO      [PASS] Output saved -> C:\Users\nirmi\Desktop\Capstone\notebooks\..\data\processed\medical_conversations_clean.csv  (165.70 MB, 274162 rows)



Saved to : C:\Users\nirmi\Desktop\Capstone\notebooks\..\data\processed\medical_conversations_clean.csv
Size     : 165.70 MB
Rows     : 274,162
Columns  : ['conversation_id', 'turn_id', 'role', 'utterance', 'source_dataset']


---
## Step 7 · Automated Validation Suite

In [10]:
log.info(DIVIDER)
log.info("  STEP 7/7 -- Automated Validation Suite")
log.info(DIVIDER)

results: List[CheckResult] = []

# ── Check 1: Zero-Null Assertion ──────────────────────────────────────────
for col in CANONICAL_COLS:
    null_count = df[col].isna().sum()
    if col == "utterance":
        blank_count = (df[col].str.strip() == "").sum()
        total = null_count + blank_count
        results.append(CheckResult(
            name   = f"Zero-Null: '{col}' (null={null_count}, blank={blank_count})",
            passed = (total == 0),
            detail = f"{total} problematic values" if total else "0 null/blank values",
        ))
    else:
        results.append(CheckResult(
            name   = f"Zero-Null: '{col}'",
            passed = (null_count == 0),
            detail = f"{null_count} null values" if null_count else "0 null values",
        ))

# ── Check 2: Turn Monotonicity ────────────────────────────────────────────
def _check_monotonicity(grp):
    turns    = grp["turn_id"].tolist()
    expected = list(range(len(turns)))
    return turns != expected

bad_mono  = (df.groupby("conversation_id", group_keys=False)
               .apply(_check_monotonicity, include_groups=False))
bad_count = bad_mono.sum()
results.append(CheckResult(
    name   = "Turn Monotonicity (starts at 0, increments by +1, no gaps/dupes)",
    passed = (bad_count == 0),
    detail = (f"{bad_count} conversations with non-monotonic turn_ids"
              if bad_count else "All conversations pass"),
))

# ── Check 3: Role Alternation ─────────────────────────────────────────────
df_sorted    = df.sort_values(["conversation_id", "turn_id"])
prev_role    = df_sorted.groupby("conversation_id")["role"].shift(1)
consec_mask  = (df_sorted["role"] == prev_role) & prev_role.notna()
consec_count = consec_mask.sum()

if consec_count > 0:
    sample_convos = df_sorted.loc[consec_mask, "conversation_id"].unique()[:5].tolist()
    alt_detail    = (f"{consec_count} consecutive same-role turns. "
                     f"Sample convos: {sample_convos}")
else:
    alt_detail = "No consecutive same-role turns detected"

results.append(CheckResult(
    name   = "Role Alternation (no consecutive user->user or asst->asst)",
    passed = (consec_count == 0),
    detail = alt_detail,
))

# ── Check 4: Conversation Boundary Integrity ──────────────────────────────
grouped     = df_sorted.groupby("conversation_id")["role"]
first_roles = grouped.first()
last_roles  = grouped.last()

bad_start = (first_roles != "user").sum()
bad_end   = (last_roles  != "assistant").sum()

results.append(CheckResult(
    name   = "Boundary: 100% conversations start with 'user'",
    passed = (bad_start == 0),
    detail = (f"{bad_start} conversations start with non-user role"
              if bad_start else "All conversations start with user"),
))
results.append(CheckResult(
    name   = "Boundary: 100% conversations end with 'assistant'",
    passed = (bad_end == 0),
    detail = (f"{bad_end} conversations end with non-assistant role"
              if bad_end else "All conversations end with assistant"),
))

# ── Check 5: Collision & Distribution Audit ───────────────────────────────
total_convos = df["conversation_id"].nunique()
total_turns  = len(df)
by_source    = (df.groupby("source_dataset")["conversation_id"]
                  .nunique().to_dict())

report.clean_conversations = total_convos
report.clean_rows          = total_turns
report.source_breakdown    = by_source

id_collisions          = df.groupby("conversation_id")["source_dataset"].nunique()
cross_source_collisions = (id_collisions > 1).sum()

dist_detail = (
    f"Total conversations={total_convos:,} | Total turns={total_turns:,} | "
    + " | ".join(f"{s}={n:,}" for s, n in by_source.items())
)
results.append(CheckResult(
    name   = "Collision Audit: conversation_id globally unique per source",
    passed = (cross_source_collisions == 0),
    detail = (f"{cross_source_collisions} cross-source ID collisions"
              if cross_source_collisions else "No cross-source collisions"),
))
results.append(CheckResult(
    name   = "Distribution Audit",
    passed = True,
    detail = dist_detail,
))

# ── Print results ─────────────────────────────────────────────────────────
all_passed = all(r.passed for r in results)

print("=" * 72)
print("  VALIDATION SUITE RESULTS")
print("=" * 72)
for r in results:
    print(r)
print("=" * 72)
overall = "[ALL CHECKS PASSED]" if all_passed else "[ONE OR MORE CHECKS FAILED]"
print(f"  Overall : {overall}")
print("=" * 72)

19:43:24  INFO      ------------------------------------------------------------------------


19:43:24  INFO        STEP 7/7 -- Automated Validation Suite


19:43:24  INFO      ------------------------------------------------------------------------


  VALIDATION SUITE RESULTS
  [PASS]  Zero-Null: 'conversation_id'
         -> 0 null values
  [PASS]  Zero-Null: 'turn_id'
         -> 0 null values
  [PASS]  Zero-Null: 'role'
         -> 0 null values
  [PASS]  Zero-Null: 'utterance' (null=0, blank=0)
         -> 0 null/blank values
  [PASS]  Zero-Null: 'source_dataset'
         -> 0 null values
  [PASS]  Turn Monotonicity (starts at 0, increments by +1, no gaps/dupes)
         -> All conversations pass
  [PASS]  Role Alternation (no consecutive user->user or asst->asst)
         -> No consecutive same-role turns detected
  [PASS]  Boundary: 100% conversations start with 'user'
         -> All conversations start with user
  [PASS]  Boundary: 100% conversations end with 'assistant'
         -> All conversations end with assistant
  [PASS]  Collision Audit: conversation_id globally unique per source
         -> No cross-source collisions
  [PASS]  Distribution Audit
         -> Total conversations=123,117 | Total turns=274,162 | Healt

---
## Audit Summary

In [11]:
print(report.banner())

------------------------------------------------------------------------
  AUDIT SUMMARY
------------------------------------------------------------------------
  Raw rows loaded          :    276,172
  Raw conversations        :    123,256

  REMOVALS
  - Null terminal asst turn (rows trimmed)  :      93
  - Null mid-turn (convos dropped)          :      37
  - Null user turn (convos dropped)         :      42
  - Asst-first convo (convos dropped)       :       0
  - Hanging user final turn (convos dropped):      60

  OUTPUT
  Clean conversations      :    123,117
  Clean rows (turns)       :    274,162

  SOURCE BREAKDOWN
    HealthChat-11k       :   10,953 conversations
    MedDialog-EN         :  112,164 conversations
------------------------------------------------------------------------
